# SUFE VOS: SAM 3.1 Object Multiplex Full Predictor

Default checkpoint source: `research21/sam3.1`. The submission path now requires official `build_sam3_multiplex_video_predictor` plus full first-frame mask conditioning. The older low-level tracker/remap path is rejected for submissions and only available as `low_level_debug` in local debugging. Recommended runtime: H100 80GB, then A100 80GB/40GB. T4 is rejected; L4 is suitable only for smoke tests.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, datetime, subprocess
DRIVE_ROOT = pathlib.Path(os.environ.get('SUFE_DRIVE_ROOT', '/content/drive/MyDrive'))
REPO_URL = os.environ.get('SUFE_REPO_URL', 'https://github.com/yyj11266/SUFE_DL_FinalProject_Vision.git')
PROJECT_ROOT = pathlib.Path(os.environ.get('SUFE_PROJECT_ROOT', '/content/sufe_vos_leaderboard')).expanduser().resolve()
if not PROJECT_ROOT.exists() and REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
drive_project = DRIVE_ROOT / 'sufe_vos_leaderboard'
if not PROJECT_ROOT.exists() and drive_project.exists():
    PROJECT_ROOT = drive_project.resolve()
DATA_ROOT = pathlib.Path(os.environ.get('SUFE_DATA_ROOT', '/content/sufe_data/video_dataset')).expanduser().resolve()
OUTPUT_ROOT = pathlib.Path(os.environ.get('SUFE_OUTPUTS_ROOT', '/content/sufe_runs')).expanduser().resolve()
REVIEW_ROOT = pathlib.Path(os.environ.get('SUFE_PUBLISH_ROOT', str(DRIVE_ROOT / 'sufe_vos_review' / 'runs'))).expanduser().resolve()
ARCHIVE_ROOT = pathlib.Path(os.environ.get('SUFE_ARCHIVE_ROOT', str(DRIVE_ROOT / 'sufe_vos_archives'))).expanduser().resolve()
HF_REPO_ID = os.environ.get('SAM31_HF_REPO_ID', 'research21/sam3.1')
CHECKPOINT_FILENAME = os.environ.get('SAM31_CHECKPOINT_FILENAME', 'sam3.1_multiplex.pt')
sample_candidates = [DRIVE_ROOT / 'sufe_vos_inputs/sample_submission.zip', PROJECT_ROOT / 'data/sample_submission.zip', pathlib.Path('/content/drive/MyDrive/sufe/sample_submission.zip')]
SAMPLE_SUBMISSION = next((path for path in sample_candidates if path.exists()), sample_candidates[0])
MOSEV2_ROOT = pathlib.Path(os.environ.get('MOSEV2_ROOT', '/content/drive/MyDrive/datasets/MOSEv2'))
checkpoint_candidates = [
    pathlib.Path(os.environ['SAM31_CHECKPOINT']) if os.environ.get('SAM31_CHECKPOINT') else None,
    DRIVE_ROOT / 'sufe_vos_inputs' / 'checkpoints' / CHECKPOINT_FILENAME,
    PROJECT_ROOT / 'checkpoints' / CHECKPOINT_FILENAME,
    OUTPUT_ROOT / 'checkpoints' / CHECKPOINT_FILENAME,
]
checkpoint_candidates.extend(sorted(OUTPUT_ROOT.glob(f'*/checkpoints/{CHECKPOINT_FILENAME}')) if OUTPUT_ROOT.exists() else [])
SAM31_CHECKPOINT = next((path for path in checkpoint_candidates if path and path.exists()), None)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REVIEW_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('MOSEV2_ROOT:', MOSEV2_ROOT)
print('HF_REPO_ID:', HF_REPO_ID)
print('SAM31_CHECKPOINT:', SAM31_CHECKPOINT if SAM31_CHECKPOINT else 'will download from Hugging Face')
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('REVIEW_ROOT:', REVIEW_ROOT)
print('ARCHIVE_ROOT:', ARCHIVE_ROOT)
def run_checked(cmd, label, exp_dir=None, env=None):
    import subprocess, json, pathlib
    print(f'Running {label}:', ' '.join(str(part) for part in cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout:
        print('--- stdout ---')
        print(result.stdout)
    if result.stderr:
        print('--- stderr ---')
        print(result.stderr)
    if exp_dir:
        status_path = pathlib.Path(exp_dir) / 'logs/per_video_status.json'
        sanity_path = pathlib.Path(exp_dir) / 'sanity_check.json'
        for path in (status_path, sanity_path):
            if path.exists():
                print(f'--- {path} ---')
                print(path.read_text(encoding='utf-8')[:12000])
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {result.returncode}. See stdout/stderr/status above.')
    return result


Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/sufe_vos_leaderboard
DATA_ROOT: /content/drive/MyDrive/sufe_vos_leaderboard/data/video_dataset
MOSEV2_ROOT: /content/drive/MyDrive/datasets/MOSEv2
HF_REPO_ID: research21/sam3.1
SAM31_CHECKPOINT: will download from Hugging Face


In [ ]:
import subprocess, sys, shutil, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements_colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'sam3'], check=False)
SAM3_REPO = pathlib.Path('/content/facebookresearch_sam3')
if SAM3_REPO.exists():
    shutil.rmtree(SAM3_REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/sam3.git', str(SAM3_REPO)], check=True)
if not (SAM3_REPO / 'sam3/model_builder.py').exists():
    raise RuntimeError(f'Official SAM3 source is incomplete: {SAM3_REPO}; files={list(SAM3_REPO.glob("sam3/*"))[:20]}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(SAM3_REPO)], check=True)
os.environ['PYTHONPATH'] = str(SAM3_REPO) + os.pathsep + os.environ.get('PYTHONPATH', '')
check_env = os.environ.copy()
subprocess.run([sys.executable, '-c', 'from sam3.model_builder import build_sam3_multiplex_video_predictor; import sam3.model_builder as mb; print("subprocess_sam3_full_predictor_ok", mb.__file__)'], check=True, env=check_env)
print('SAM3 source:', SAM3_REPO)
print('PYTHONPATH head:', os.environ['PYTHONPATH'].split(os.pathsep)[0])


SAM3 source: /content/facebookresearch_sam3
PYTHONPATH head: /content/facebookresearch_sam3


In [ ]:
import subprocess, sys, os, platform
runtime_code = r'''
import torch, platform
from packaging.version import Version
from sam3.model_builder import build_sam3_multiplex_video_predictor
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), torch.__version__
assert torch.cuda.is_available(), 'CUDA GPU is required'
assert Version(str(torch.version.cuda)) >= Version('12.6'), torch.version.cuda
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
assert major >= 8 and torch.cuda.is_bf16_supported(), f'BF16 Ampere+ GPU required, got {gpu_name}'
assert 'T4' not in gpu_name.upper(), 'T4 is excluded for this full run'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu_name, 'sam3_builder': 'build_sam3_multiplex_video_predictor'})
'''
run_checked([sys.executable, '-c', runtime_code], 'runtime-preflight', env=os.environ.copy())
if SAM31_CHECKPOINT:
    checkpoint = str(SAM31_CHECKPOINT)
else:
    download_env = os.environ.copy()
    download_env['HF_REPO_ID'] = HF_REPO_ID
    download_env['CHECKPOINT_FILENAME'] = CHECKPOINT_FILENAME
    checkpoint = subprocess.check_output([sys.executable, '-c', 'import os; from huggingface_hub import hf_hub_download; print(hf_hub_download(os.environ["HF_REPO_ID"], os.environ["CHECKPOINT_FILENAME"]))'], text=True, env=download_env).strip().splitlines()[-1]
print('checkpoint:', checkpoint)


Running runtime-preflight: /usr/bin/python3 -c 
import torch, platform
from packaging.version import Version
from sam3.model_builder import build_sam3_multiplex_video_predictor
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), torch.__version__
assert torch.cuda.is_available(), 'CUDA GPU is required'
assert Version(str(torch.version.cuda)) >= Version('12.6'), torch.version.cuda
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
assert major >= 8 and torch.cuda.is_bf16_supported(), f'BF16 Ampere+ GPU required, got {gpu_name}'
assert 'T4' not in gpu_name.upper(), 'T4 is excluded for this full run'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu_name, 'sam3_builder': 'build_sam3_multiplex_video_predictor'})

--- stdout ---
{'python': '3.12.13', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA L4', 'sam3_builder': 'build_sam3_multiplex_video_predictor'}

--- stder

In [4]:
import json
from IPython.display import Image as DisplayImage, display
smoke_videos = ['0u8fy7u2', '2b827e3a', '2a1jkxdf', 'kpg9gld7', 'lkob5diu', 'pjlde9hu']
SMOKE_EXP = 'sam31_smoke_full_predictor_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_sam31_vos.py'),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', SMOKE_EXP,
    '--checkpoint', checkpoint,
    '--sam3-repo-dir', str(SAM3_REPO),
    '--prompt-mode', 'mask',
    '--sam3-run-mode', 'full_predictor_mask',
    '--original-resolution',
    '--video-ids', ','.join(smoke_videos),
    '--max-frames', '20',
    '--save-native-scores',
    '--save-overlays', 'all',
    '--smoke-quality-gate',
]
smoke_dir = OUTPUT_ROOT / SMOKE_EXP
run_checked(cmd, 'smoke-full-predictor', exp_dir=smoke_dir, env=os.environ.copy())
smoke_gate = json.loads((smoke_dir / 'logs/smoke_quality_gate.json').read_text())
assert smoke_gate['passed'], smoke_gate
print(json.dumps({'smoke_exp': SMOKE_EXP, 'warnings': smoke_gate.get('warnings', []), 'contact_sheets': smoke_gate.get('contact_sheets', {})}, indent=2))
for path in smoke_gate.get('contact_sheets', {}).values():
    display(DisplayImage(filename=path))


Running smoke-full-predictor: /usr/bin/python3 /content/drive/MyDrive/sufe_vos_leaderboard/scripts/run_sam31_vos.py --data-root /content/drive/MyDrive/sufe_vos_leaderboard/data/video_dataset --output-dir /content/drive/MyDrive/sufe_vos_leaderboard/outputs --experiment-id sam31_smoke_full_predictor_20260617_153938 --checkpoint /root/.cache/huggingface/hub/models--research21--sam3.1/snapshots/3bd4a45137d9db4cb21c0119e7eb67deb69e339f/sam3.1_multiplex.pt --sam3-repo-dir /content/facebookresearch_sam3 --prompt-mode mask --sam3-run-mode full_predictor_mask --original-resolution --video-ids 0u8fy7u2,2b827e3a,2a1jkxdf,kpg9gld7,lkob5diu,pjlde9hu --max-frames 20 --save-native-scores --smoke-quality-gate
--- stdout ---
{"stage": "prepared_data", "num_selected_videos": 6, "selected_videos": ["0u8fy7u2", "2b827e3a", "2a1jkxdf", "kpg9gld7", "lkob5diu", "pjlde9hu"], "experiment_dir": "/content/drive/MyDrive/sufe_vos_leaderboard/outputs/sam31_smoke_full_predictor_20260617_153938"}
{"stage": "checkpoin

RuntimeError: smoke-full-predictor failed with exit code 1. See stdout/stderr/status above.

In [ ]:
SPLIT_PATH = OUTPUT_ROOT / 'validation/mosev2_seed2026.json'
if MOSEV2_ROOT.exists():
    split_cmd = [sys.executable, '-m', 'src.eval.mosev2_split', '--root', str(MOSEV2_ROOT), '--output', str(SPLIT_PATH), '--seed', '2026', '--total', '80', '--calibration-size', '40']
    run_checked(split_cmd, 'mosev2-split', env=os.environ.copy())
    for split_name in ('calibration', 'holdout'):
        exp = f'sam31_mosev2_{split_name}_seed2026'
        split_run = [sys.executable, str(PROJECT_ROOT / 'scripts/run_sam31_vos.py'), '--data-root', str(MOSEV2_ROOT), '--output-dir', str(OUTPUT_ROOT), '--experiment-id', exp, '--checkpoint', checkpoint, '--sam3-repo-dir', str(SAM3_REPO), '--prompt-mode', 'mask', '--sam3-run-mode', 'full_predictor_mask', '--original-resolution', '--video-ids-file', str(SPLIT_PATH), '--split', split_name, '--save-native-scores', '--save-overlays', 'sample', '--skip-existing']
        split_dir = OUTPUT_ROOT / exp
        run_checked(split_run, f'mosev2-{split_name}', exp_dir=split_dir, env=os.environ.copy())
else:
    print('MOSEv2 not found; place it at', MOSEV2_ROOT, 'before threshold calibration. SUFE baseline can still run.')


In [ ]:
import json
smoke_gate = json.loads((smoke_dir / 'logs/smoke_quality_gate.json').read_text())
assert smoke_gate['passed'], smoke_gate
if not SAMPLE_SUBMISSION.exists():
    raise RuntimeError(f'SAMPLE_SUBMISSION is required for strict validation before submission: {SAMPLE_SUBMISSION}')
FULL_EXP = 'sam31_native_full_predictor_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
full_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/run_sam31_vos.py'),
    '--data-root', str(DATA_ROOT),
    '--sample-submission', str(SAMPLE_SUBMISSION),
    '--output-dir', str(OUTPUT_ROOT),
    '--experiment-id', FULL_EXP,
    '--checkpoint', checkpoint,
    '--sam3-repo-dir', str(SAM3_REPO),
    '--prompt-mode', 'mask',
    '--sam3-run-mode', 'full_predictor_mask',
    '--original-resolution',
    '--save-native-scores',
    '--save-overlays', 'sample',
    '--skip-existing',
    '--quality-gate',
    '--quality-gate-baseline-exp', os.environ.get('SAM31_QUALITY_GATE_BASELINE_EXP', str(OUTPUT_ROOT / 'data_prep_20260608_065509')),
    '--make-submission',
]
full_dir = OUTPUT_ROOT / FULL_EXP
run_checked(full_cmd, 'full-predictor', exp_dir=full_dir, env=os.environ.copy())


publish_dir = REVIEW_ROOT / FULL_EXP
publish_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts/publish_run_for_codex.py'),
    '--exp-dir', str(full_dir),
    '--publish-dir', str(publish_dir),
    '--data-root', str(DATA_ROOT),
    '--archive-dir', str(ARCHIVE_ROOT),
    '--replace',
]
if os.environ.get('SUFE_MAKE_FULL_ARCHIVE', '0').strip().lower() in {'1', 'true', 'yes'}:
    publish_cmd.append('--make-full-archive')
run_checked(publish_cmd, 'publish-codex-review', exp_dir=publish_dir, env=os.environ.copy())
print('Codex review bundle:', publish_dir)


In [ ]:
import json, zipfile
exp_dir = OUTPUT_ROOT / FULL_EXP
status = json.loads((exp_dir / 'logs/per_video_status.json').read_text())
sanity = json.loads((exp_dir / 'sanity_check.json').read_text())
manifest = json.loads((exp_dir / 'run_manifest.json').read_text())
assert manifest['sam3_run_mode'] == 'full_predictor_mask', manifest
assert manifest['mask_conditioning_path'] == 'official_full_predictor_private_tracker_add_new_objects', manifest
assert status['summary']['status'] == 'done', status['summary']
assert status['summary']['num_videos'] == 25, status['summary']
assert status['summary']['num_frames'] == 2015, status['summary']
assert status['summary']['all_first_frames_exact'] is True
assert sanity['passed'] and sanity['num_actual_masks'] == 2015, sanity
with zipfile.ZipFile(exp_dir / 'submission.zip') as archive:
    assert len([name for name in archive.namelist() if name.lower().endswith('.png')]) == 2015
print({'experiment': FULL_EXP, 'submission': str(exp_dir / 'submission.zip'), 'sanity': sanity, 'manifest_mode': manifest['sam3_run_mode']})
